# RAG Chat — Dukcapil

Notebook ini dijalankan **berkali-kali** untuk eksperimen retrieval & generation. Vector store sudah ter-persist dari `build_vectorstore.ipynb`.

**Empat retrieval variants** (cell terpisah, gampang di-swap):
- **V1 — Naive**: similarity search top-k
- **V2 — + Reranker**: similarity top-20 → bge-reranker-v2-m3 → top-k
- **V3 — Hybrid**: BM25 + Dense (RRF)
- **V4 — Hybrid + Reranker**: BM25 + Dense → rerank → top-k

Pakai `chat(query, retriever_fn=retrieve_v1)` untuk eksperimen tiap variant.

## Step 1 — Load Vector Store

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert api_key, "GEMINI_API_KEY tidak ditemukan di .env"
os.environ["GOOGLE_API_KEY"] = api_key
print("API key OK")

API key OK


In [2]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# PENTING: task_type="retrieval_query" untuk embedding query (beda dengan saat embedding dokumen)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",  # ← BENAR
    task_type="retrieval_query",
    output_dimensionality=768,  # Harus sama dengan build_vectorstore.ipynb
)

vectorstore = Chroma(
    collection_name="dukcapil_qa",
    embedding_function=embeddings,
    persist_directory="../data/dukcapil_vector_store",
)

n_chunks = vectorstore._collection.count()
print(f"Loaded {n_chunks} chunks dari vector store")
assert n_chunks > 0, "Vector store kosong — jalankan build_vectorstore.ipynb dulu!"

c:\Users\Nafisha\Documents\RAGTrial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Loaded 150 chunks dari vector store


In [3]:
# Load semua chunks ke memori sebagai LangChain Documents (untuk BM25 di V3/V4)
from langchain_core.documents import Document

raw = vectorstore.get()
all_chunks = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(raw['documents'], raw['metadatas'])
]
print(f"Loaded {len(all_chunks)} chunks in-memory untuk BM25")

Loaded 150 chunks in-memory untuk BM25


## Step 2 — Setup LLM (Gemini) + Prompt Template

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_tokens=1024,
)

# Smoke test
test = llm.invoke("Jawab dengan satu kata: ibu kota Indonesia?")
print(f"LLM OK. Response: {test.content}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


LLM OK. Response: Jakarta


In [5]:
PROMPT_TEMPLATE = """Kamu adalah asisten yang membantu menjawab pertanyaan seputar administrasi kependudukan dan pencatatan sipil di Indonesia, berdasarkan Buku Saku Dukcapil 2023.

ATURAN:
1. Jawab HANYA berdasarkan konteks di bawah. Jangan menambah informasi dari pengetahuan umum.
2. Jika informasi tidak ada dalam konteks, jawab: "Informasi tidak ditemukan dalam buku saku."
3. Sebutkan sumber (BAB & halaman) di akhir jawaban dalam format: [Sumber: <section>, hal <page>]
4. Jawab dalam Bahasa Indonesia yang jelas dan ringkas.

KONTEKS:
{context}

PERTANYAAN: {question}

JAWABAN:"""

## Step 3 — Empat Retrieval Variants

Tiap variant adalah function `retrieve_vX(query, k=5)` yang return `List[Document]`. Bisa di-swap di `chat()`.

### Variant 1 — Naive Similarity (baseline)

In [6]:
def retrieve_v1(query, k=5):
    return vectorstore.similarity_search(query, k=k)

# Test
results = retrieve_v1("Apa syarat penerbitan KTP-el pertama kali bagi WNI?", k=3)
for r in results:
    print(f"- {r.metadata.get('section','?')[:25]} | hal {r.metadata.get('page_start')} | Q#{r.metadata.get('question_number')}")
    print(f"  {r.page_content[:120]}...")

- BAB II - Pertanyaan dan J | hal 40 | Q#1
  1. Bagaimana penerbitan KTP-el pertama kali bagi WNI?
Jawaban:
Berdasarkan Pasal 15 Perpres Nomor
96 Tahun 2018, penerbi...
- BAB II - Pertanyaan dan J | hal 42 | Q#4
  4. Apakah penerbitan KTP-el dapat dilakukan di luar Kabupaten/Kota alamat domisili yang tertera dalam KKnya?
Jawaban:
Be...
- BAB II - Pertanyaan dan J | hal 40 | Q#2
  2. Apakah Orang Asing boleh memiliki KTP-el?
Jawaban:
Berdasarkan Pasal 16 Perpres Nomor
96 Tahun 2018, orang asing bole...


### Variant 2 — Similarity + Cross-Encoder Reranker (BAAI/bge-reranker-v2-m3)

Pertama-tama load reranker (~600MB, download otomatis sekali). Lalu fetch top-20 dari similarity → rerank → top-k.

In [7]:
from sentence_transformers import CrossEncoder

# Load sekali, cache di module-level
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")
print("Reranker loaded")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 439.78it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]              


Reranker loaded


In [8]:
def retrieve_v2(query, k=5, fetch_k=20):
    candidates = vectorstore.similarity_search(query, k=fetch_k)
    pairs = [(query, c.page_content) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)
    return [c for c, _ in ranked[:k]]

# Test
results = retrieve_v2("Apa syarat penerbitan KTP-el pertama kali bagi WNI?", k=3)
for r in results:
    print(f"- {r.metadata.get('section','?')[:25]} | hal {r.metadata.get('page_start')} | Q#{r.metadata.get('question_number')}")
    print(f"  {r.page_content[:120]}...")

- BAB II - Pertanyaan dan J | hal 40 | Q#1
  1. Bagaimana penerbitan KTP-el pertama kali bagi WNI?
Jawaban:
Berdasarkan Pasal 15 Perpres Nomor
96 Tahun 2018, penerbi...
- BAB II - Pertanyaan dan J | hal 43 | Q#5
  5. Apa perbedaan antara KTP-el WNI dan KTP-el WNA?
Jawaban:
Perbedaan:
a. KTP-el bagi WNI berwarna biru gradasi sedangka...
- BAB II - Pertanyaan dan J | hal 64 | Q#7
  7. Bagaimana cara melaporkan penduduk WNI yang pindah ke luar negeri?
Jawaban:
Berdasarkan ketentutan Pasal 28 ayat (2) ...


### Variant 3 — Hybrid (BM25 + Dense, dengan Reciprocal Rank Fusion)

BM25 menangkap keyword exact match (mis. "KTP-el", "SPTJM") sementara dense embedding menangkap semantik. Combine pakai EnsembleRetriever (RRF default).

In [9]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(all_chunks)
bm25.k = 10

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

ensemble = EnsembleRetriever(
    retrievers=[bm25, dense_retriever],
    weights=[0.5, 0.5],
)

print("Hybrid retriever ready")

Hybrid retriever ready


In [10]:
def retrieve_v3(query, k=5):
    return ensemble.invoke(query)[:k]

# Test
results = retrieve_v3("Apa syarat penerbitan KTP-el pertama kali bagi WNI?", k=3)
for r in results:
    print(f"- {r.metadata.get('section','?')[:25]} | hal {r.metadata.get('page_start')} | Q#{r.metadata.get('question_number')}")
    print(f"  {r.page_content[:120]}...")

- BAB II - Pertanyaan dan J | hal 40 | Q#1
  1. Bagaimana penerbitan KTP-el pertama kali bagi WNI?
Jawaban:
Berdasarkan Pasal 15 Perpres Nomor
96 Tahun 2018, penerbi...
- BAB II - Pertanyaan dan J | hal 43 | Q#5
  5. Apa perbedaan antara KTP-el WNI dan KTP-el WNA?
Jawaban:
Perbedaan:
a. KTP-el bagi WNI berwarna biru gradasi sedangka...
- BAB II - Pertanyaan dan J | hal 42 | Q#4
  4. Apakah penerbitan KTP-el dapat dilakukan di luar Kabupaten/Kota alamat domisili yang tertera dalam KKnya?
Jawaban:
Be...


### Variant 4 — Hybrid + Reranker (full stack)

Ensemble fetch top-20 → rerank pakai bge-reranker → top-k. Kombinasi terbaik kalau aware dengan trade-off latency.

In [11]:
def retrieve_v4(query, k=5, fetch_k=20):
    bm25.k = fetch_k
    dense_retriever.search_kwargs["k"] = fetch_k
    candidates = ensemble.invoke(query)
    pairs = [(query, c.page_content) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)
    return [c for c, _ in ranked[:k]]

# Test
results = retrieve_v4("Apa syarat penerbitan KTP-el pertama kali bagi WNI?", k=3)
for r in results:
    print(f"- {r.metadata.get('section','?')[:25]} | hal {r.metadata.get('page_start')} | Q#{r.metadata.get('question_number')}")
    print(f"  {r.page_content[:120]}...")

- BAB II - Pertanyaan dan J | hal 40 | Q#1
  1. Bagaimana penerbitan KTP-el pertama kali bagi WNI?
Jawaban:
Berdasarkan Pasal 15 Perpres Nomor
96 Tahun 2018, penerbi...
- BAB II - Pertanyaan dan J | hal 53 | Q#2
  2. Bagaimana cara mengurus pindah penduduk antar Kabupaten/Kota?
Jawaban:
Berdasarkan ketentutan Pasal 25 ayat (3) Perpr...
- BAB II - Pertanyaan dan J | hal 51 | Q#1
  1. Bagaimana cara mengurus pindah penduduk berbeda kelurahan/ kecamatan namun masih dalam 1 (satu) kabupaten/kota?
Jawab...


## Step 4 — `chat()` Function

Wrapper yang gabungkan retrieval + LLM. Swap `retriever_fn` untuk coba variant berbeda.

In [12]:
def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        meta = d.metadata
        section = meta.get('section', '?')
        page = meta.get('page_start', '?')
        q_info = f", Q#{meta.get('question_number')}" if meta.get('question_number') else ""
        header = f"[Sumber {i}: {section}, hal {page}{q_info}]"
        parts.append(f"{header}\n{d.page_content}")
    return "\n\n---\n\n".join(parts)


def chat(query, retriever_fn=None, k=5, show_sources=True, verbose=False):
    if retriever_fn is None:
        retriever_fn = retrieve_v1
    docs = retriever_fn(query, k=k)
    context = format_context(docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=query)
    if verbose:
        print("=== PROMPT ===")
        print(prompt[:2000])
        print("...\n")
    answer = llm.invoke(prompt).content
    result = {"answer": answer}
    if show_sources:
        result["sources"] = [
            {
                "section": d.metadata.get('section'),
                "page": d.metadata.get('page_start'),
                "question_number": d.metadata.get('question_number'),
                "subsection": d.metadata.get('subsection'),
            }
            for d in docs
        ]
    return result

# Smoke test pakai V1
result = chat("Apa syarat penerbitan KTP-el pertama kali bagi WNI?", retriever_fn=retrieve_v1, k=5)
print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s}")

ANSWER:
Berdasarkan Pasal 15 Perpres Nomor 96 Tahun 2018, persyaratan penerbitan KTP-el pertama kali bagi WNI adalah:
a. Telah berusia 17 tahun, sudah kawin atau pernah kawin; dan
b. Fotokopi KK.
[Sumber: BAB II - Pertanyaan dan Jawaban, hal 40]

SOURCES:
  - {'section': 'BAB II - Pertanyaan dan Jawaban', 'page': 40, 'question_number': 1, 'subsection': 'Unknown'}
  - {'section': 'BAB II - Pertanyaan dan Jawaban', 'page': 42, 'question_number': 4, 'subsection': 'Unknown'}
  - {'section': 'BAB II - Pertanyaan dan Jawaban', 'page': 40, 'question_number': 2, 'subsection': 'Unknown'}
  - {'section': 'BAB II - Pertanyaan dan Jawaban', 'page': 43, 'question_number': 5, 'subsection': 'Unknown'}
  - {'section': 'BAB II - Pertanyaan dan Jawaban', 'page': 44, 'question_number': 6, 'subsection': 'Unknown'}


## Step 5 — Test Harness: Compare All 4 Variants

Loop sample queries × 4 variants → bandingkan side-by-side untuk evaluasi kualitatif.

In [13]:
TEST_QUERIES = [
    # Literal match (BAB II)
    "Apa syarat penerbitan KTP-el pertama kali bagi WNI?",
    "Apakah NIK yang tidak sesuai dengan format tanggal lahir dapat diubah?",
    # Parafrasa (semantic match)
    "Kalau warga negara asing tinggal tetap di Indonesia, apakah bisa punya e-KTP?",
    "Bagaimana prosedur pindah domisili untuk WNA pemegang KITAP?",
    # Narrative (BAB I/III)
    "Apa latar belakang penyusunan buku saku ini?",
    # Edge case: out-of-scope
    "Siapa presiden Indonesia tahun 2024?",
]

VARIANTS = {
    "V1 (Naive)":    retrieve_v1,
    "V2 (+Rerank)":  retrieve_v2,
    "V3 (Hybrid)":   retrieve_v3,
    "V4 (Hyb+Rerank)": retrieve_v4,
}

In [14]:
# Tip: jalankan cell ini SEKALI dan cek output. Hati-hati API rate limit Gemini.
# Total: len(TEST_QUERIES) * len(VARIANTS) = 24 call ke Gemini.

for query in TEST_QUERIES:
    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)
    for name, fn in VARIANTS.items():
        try:
            result = chat(query, retriever_fn=fn, k=5, show_sources=True)
            print(f"\n--- {name} ---")
            print(result["answer"])
            srcs = ", ".join(
                f"{s.get('section','?')[:8]}/h{s.get('page')}/Q{s.get('question_number') or '-'}"
                for s in result["sources"]
            )
            print(f"  sources: {srcs}")
        except Exception as e:
            print(f"\n--- {name} ---  ERROR: {e}")
    print()

QUERY: Apa syarat penerbitan KTP-el pertama kali bagi WNI?

--- V1 (Naive) ---
Syarat penerbitan KTP-el pertama kali bagi WNI adalah:
a. Telah berusia 17 tahun, sudah kawin atau pernah kawin; dan
b. Fotokopi KK.

[Sumber: BAB II - Pertanyaan dan Jawaban, hal 40]
  sources: BAB II -/h40/Q1, BAB II -/h42/Q4, BAB II -/h40/Q2, BAB II -/h43/Q5, BAB II -/h44/Q6

--- V2 (+Rerank) ---
Syarat penerbitan KTP-el pertama kali bagi WNI adalah:
a. Telah berusia 17 tahun, sudah kawin atau pernah kawin; dan
b. Fotokopi KK.
[Sumber: BAB II - Pertanyaan dan Jawaban, hal 40]
  sources: BAB II -/h40/Q1, BAB II -/h43/Q5, BAB II -/h64/Q7, BAB II -/h66/Q8, BAB II -/h42/Q4

--- V3 (Hybrid) ---
Syarat penerbitan KTP-el pertama kali bagi WNI adalah:
a. Telah berusia 17 tahun, sudah kawin atau pernah kawin; dan
b. Fotokopi KK.

[Sumber: BAB II - Pertanyaan dan Jawaban, hal 40]
  sources: BAB II -/h40/Q1, BAB II -/h43/Q5, BAB II -/h42/Q4, BAB II -/h44/Q6, BAB II -/h269/Q4

--- V4 (Hyb+Rerank) ---
Berdasarkan Pasa

## Step 6 — Interactive Single-Query Cell

Edit `MY_QUERY` dan `MY_VARIANT` untuk eksperimen cepat.

In [15]:
MY_QUERY = "Apa syarat penerbitan KTP-el pertama kali bagi WNI?"
MY_VARIANT = retrieve_v4   # ganti ke retrieve_v1 / v2 / v3 / v4

result = chat(MY_QUERY, retriever_fn=MY_VARIANT, k=5, verbose=False)
print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 7.852391122s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [16]:
# debug

from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Load vectorstore
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_query",
    output_dimensionality=768,
)

vectorstore = Chroma(
    collection_name="dukcapil_qa",
    embedding_function=embeddings,
    persist_directory="../data/dukcapil_vector_store",
)

# Retrieve tanpa LLM, lihat chunks apa yang keluar
query = "Apa syarat penerbitan KTP-el pertama kali bagi WNI?"
print(f"Query: {query}\n")

results = vectorstore.similarity_search(query, k=5)
print(f"Retrieved {len(results)} chunks:\n")

for i, r in enumerate(results, 1):
    print(f"[{i}] {r.metadata.get('section')} | hal {r.metadata.get('page_start')} | Q#{r.metadata.get('question_number')}")
    print(f"    Content: {r.page_content[:200]}...\n")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Query: Apa syarat penerbitan KTP-el pertama kali bagi WNI?

Retrieved 5 chunks:

[1] BAB II - Pertanyaan dan Jawaban | hal 40 | Q#1
    Content: 1. Bagaimana penerbitan KTP-el pertama kali bagi WNI?
Jawaban:
Berdasarkan Pasal 15 Perpres Nomor
96 Tahun 2018, penerbitan KTP-el bagi penduduk WNI harus memenuhi persyaratan:
a. Telah berusia 17 tah...

[2] BAB II - Pertanyaan dan Jawaban | hal 42 | Q#4
    Content: 4. Apakah penerbitan KTP-el dapat dilakukan di luar Kabupaten/Kota alamat domisili yang tertera dalam KKnya?
Jawaban:
Berdasarkan ketentuan Pasal 
 Permendagri Nomor 8 Tahun 2016, bahwa penerbitan KTP...

[3] BAB II - Pertanyaan dan Jawaban | hal 40 | Q#2
    Content: 2. Apakah Orang Asing boleh memiliki KTP-el?
Jawaban:
Berdasarkan Pasal 16 Perpres Nomor
96 Tahun 2018, orang asing boleh memiliki KTP-el jika memiliki Izin Tinggal tetap dan

\,
' ' terdaftar sebagai...

[4] BAB II - Pertanyaan dan Jawaban | hal 43 | Q#5
    Content: 5. Apa perbedaan antara KTP-el WNI dan KTP-el WN